# 实践实验——应用机器学习的建议
在本实验中，你将探索评估和改进机器学习模型的技术。

# 大纲
- [ 1 - 软件包 ](#1)
- [ 2 - 评估学习算法（多项式回归）](#2)
  - [ 2.1 拆分数据集](#2.1)
  - [ 2.2 用于模型评估的误差计算：线性回归](#2.2)
    - [ 练习 1](#ex01)
  - [ 2.3 比较训练数据与测试数据上的表现](#2.3)
- [ 3 - 偏差与方差<img align="Right" src="./images/C2_W3_BiasVarianceDegree.png"  style=" width:500px; padding: 10px 20px ; "> ](#3)
  - [ 3.1 绘制训练集、交叉验证集、测试集](#3.1)
  - [ 3.2 寻找最优次数](#3.2)
  - [ 3.3 调整正则化。](#3.3)
  - [ 3.4 获取更多数据：增加训练集大小 (m)](#3.4)
- [ 4 - 评估学习算法（神经网络）](#4)
  - [ 4.1 数据集](#4.1)
  - [ 4.2 通过计算分类误差评估类别模型](#4.2)
    - [ 练习 2](#ex02)
- [ 5 - 模型复杂度](#5)
  - [ 练习 3](#ex03)
  - [ 5.1 简单模型](#5.1)
    - [ 练习 4](#ex04)
- [ 6 - 正则化](#6)
  - [ 练习 5](#ex05)
- [ 7 - 通过迭代寻找最优正则化值](#7)
  - [ 7.1 测试](#7.1)

<a name="1"></a>
## 1——软件包

首先，运行下面的单元格，导入本作业所需的全部软件包。
- [numpy](https://numpy.org/) 是 Python 科学计算的基础软件包。
- [matplotlib](http://matplotlib.org) 是 Python 中常用的绘图库。
- [scikitlearn](https://scikit-learn.org/stable/) 是一个基础的数据挖掘库。
- [tensorflow](https://www.tensorflow.org/) 是常用的机器学习平台。

In [ ]:
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.activations import relu,linear
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.optimizers import Adam

import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)

from public_tests_a1 import * 

tf.keras.backend.set_floatx('float64')
from assigment_utils import *

tf.autograph.set_verbosity(0)

<a name="2"></a>
## 2 - 评估学习算法（多项式回归）

<img align="Right" src="./images/C2_W3_TrainingVsNew.png"  style=" width:350px; padding: 10px 20px ; "> 假设你创建了一个机器学习模型，并发现它对训练数据的*拟合*非常好。这样就完成了吗？还没有。创建模型的目标，是能够为 <span style="color:blue">*新的*</span> 样本预测数值。

在部署模型之前，如何测试它在新数据上的性能？  
答案分为两部分：
* 将原始数据集拆分为“训练”集和“测试”集。
    * 使用训练数据拟合模型参数
    * 使用测试数据评估模型在*新*数据上的表现
* 构造一个误差函数来评估模型。

<a name="2.1"></a>
### 2.1 划分数据集
课程建议保留数据集的 20%～40% 用于测试。让我们使用 `sklearn` 函数 [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) 来执行划分。运行下面的单元格后，请再次核对各数组的形状。

In [ ]:
# Generate some data
X,y,x_ideal,y_ideal = gen_data(18, 2, 0.7)
print("X.shape", X.shape, "y.shape", y.shape)

#split the data using sklearn routine 
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.33, random_state=1)
print("X_train.shape", X_train.shape, "y_train.shape", y_train.shape)
print("X_test.shape", X_test.shape, "y_test.shape", y_test.shape)

#### 2.1.1 绘制训练集和测试集
从下图可以看到，将作为训练数据的点（红色）与模型未在其上训练的点（测试数据）混合在一起。这个特定数据集是一个添加了噪声的二次函数。图中同时显示了“理想”曲线以供参考。

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(4,4))
ax.plot(x_ideal, y_ideal, "--", color = "orangered", label="y_ideal", lw=1)
ax.set_title("Training, Test",fontsize = 14)
ax.set_xlabel("x")
ax.set_ylabel("y")

ax.scatter(X_train, y_train, color = "red",           label="train")
ax.scatter(X_test, y_test,   color = dlc["dlblue"],   label="test")
ax.legend(loc='upper left')
plt.show()

<a name="2.2"></a>
### 2.2 用于模型评估的误差计算：线性回归
在*评估*线性回归模型时，需要对预测值与目标值之差的平方误差取平均值。

$$ J_\text{test}(\mathbf{w},b) = 
            \frac{1}{2m_\text{test}}\sum_{i=0}^{m_\text{test}-1} ( f_{\mathbf{w},b}(\mathbf{x}^{(i)}_\text{test}) - y^{(i)}_\text{test} )^2 
            \tag{1}
$$

<a name="ex01"></a>
### 练习 1

在下方创建一个函数，用于评估线性回归模型在数据集上的误差。

In [ ]:
# UNQ_C1
# GRADED CELL: eval_mse
def eval_mse(y, yhat):
    """ 
    Calculate the mean squared error on a data set.
    Args:
      y    : (ndarray  Shape (m,) or (m,1))  target value of each example
      yhat : (ndarray  Shape (m,) or (m,1))  predicted value of each example
    Returns:
      err: (scalar)             
    """
    m = len(y)
    err = 0.0
    for i in range(m):
    ### START CODE HERE ### 
    
    ### END CODE HERE ### 
    
    return(err)

In [ ]:
y_hat = np.array([2.4, 4.2])
y_tmp = np.array([2.3, 4.1])
eval_mse(y_hat, y_tmp)

# BEGIN UNIT TEST
test_eval_mse(eval_mse)   
# END UNIT TEST

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>

    
```python
def eval_mse(y, yhat):
    """ 
    Calculate the mean squared error on a data set.
    Args:
      y    : (ndarray  Shape (m,) or (m,1))  target value of each example
      yhat : (ndarray  Shape (m,) or (m,1))  predicted value of each example
    Returns:
      err: (scalar)             
    """
    m = len(y)
    err = 0.0
    for i in range(m):
        err_i  = ( (yhat[i] - y[i])**2 ) 
        err   += err_i                                                                
    err = err / (2*m)                    
    return(err)
```

<a name="2.3"></a>
### 2.3 比较训练数据与测试数据上的性能
让我们构建一个高次多项式模型，以最小化训练误差。这里将使用 `sklearn` 中的 linear_regression 函数。如果想查看详细信息，代码位于导入的实用工具文件中。以下步骤为：
* 创建并拟合模型。（“拟合”是训练或运行梯度下降的另一种说法。）
* 计算训练数据上的误差。
* 计算测试数据上的误差。

In [ ]:
# create a model in sklearn, train on training data
degree = 10
lmodel = lin_model(degree)
lmodel.fit(X_train, y_train)

# predict on training data, find training error
yhat = lmodel.predict(X_train)
err_train = lmodel.mse(y_train, yhat)

# predict on test data, find error
yhat = lmodel.predict(X_test)
err_test = lmodel.mse(y_test, yhat)

训练集上计算出的误差明显小于测试集上的误差。

In [ ]:
print(f"training err {err_train:0.2f}, test err {err_test:0.2f}")

下图说明了原因。模型对训练数据拟合得非常好，为此它创建了一个复杂函数。测试数据并未参与训练，而模型在这些数据上的预测表现很差。  
可以这样描述此模型：1）过拟合；2）方差高；3）“泛化”能力差。

In [ ]:
# plot predictions over data range 
x = np.linspace(0,int(X.max()),100)  # predict values for plot
y_pred = lmodel.predict(x).reshape(-1,1)

plt_train_test(X_train, y_train, X_test, y_test, x, y_pred, x_ideal, y_ideal, degree)

测试集误差表明，该模型无法很好地用于新数据。如果使用测试误差来指导模型改进，模型确实会在测试数据上表现良好……但测试数据原本是用来代表*新*数据的。
因此，还需要另一组数据来检验模型在新数据上的表现。

课程中提出的方案是将数据分成三组。下表所示的训练集、交叉验证集和测试集比例是一种典型分配，但也可以根据可用数据量进行调整。

| 数据             | 占总量百分比 | 描述 |
|------------------|:----------:|:---------|
| 训练集         | 60         | 在训练或拟合期间用于调整模型参数 $w$ 和 $b$ 的数据 |
| 交叉验证集 | 20         | 用于调整其他模型参数的数据，例如多项式次数、正则化或神经网络架构。|
| 测试集             | 20         | 调参完成后用于测试模型，以衡量其在新数据上表现的数据 |


下面生成三个数据集。我们将再次使用 `sklearn` 中的 `train_test_split`，但会调用两次以获得三份数据：

In [ ]:
# Generate  data
X,y, x_ideal,y_ideal = gen_data(40, 5, 0.7)
print("X.shape", X.shape, "y.shape", y.shape)

#split the data using sklearn routine 
X_train, X_, y_train, y_ = train_test_split(X,y,test_size=0.40, random_state=1)
X_cv, X_test, y_cv, y_test = train_test_split(X_,y_,test_size=0.50, random_state=1)
print("X_train.shape", X_train.shape, "y_train.shape", y_train.shape)
print("X_cv.shape", X_cv.shape, "y_cv.shape", y_cv.shape)
print("X_test.shape", X_test.shape, "y_test.shape", y_test.shape)

<a name="3"></a>
## 3 - 偏差与方差<img align="Right" src="./images/C2_W3_BiasVarianceDegree.png"  style=" width:500px; padding: 10px 20px ; "> 
上面很明显可以看出，多项式模型的次数过高。如何选择一个合适的值？事实证明，如图所示，训练性能和交叉验证性能可以提供指导。通过尝试一系列次数值，可以评估训练性能和交叉验证性能。当次数变得过大时，相对于训练性能，交叉验证性能会开始下降。让我们在示例中尝试一下。

<a name="3.1"></a>
### 3.1 绘制训练集、交叉验证集和测试集
从下图可以看到，属于训练集的数据点（红色）与模型未用于训练的数据点（测试集和交叉验证集）混合在一起。

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(4,4))
ax.plot(x_ideal, y_ideal, "--", color = "orangered", label="y_ideal", lw=1)
ax.set_title("Training, CV, Test",fontsize = 14)
ax.set_xlabel("x")
ax.set_ylabel("y")

ax.scatter(X_train, y_train, color = "red",           label="train")
ax.scatter(X_cv, y_cv,       color = dlc["dlorange"], label="cv")
ax.scatter(X_test, y_test,   color = dlc["dlblue"],   label="test")
ax.legend(loc='upper left')
plt.show()

<a name="3.2"></a>
### 3.2 寻找最优次数
在之前的实验中，你发现可以利用多项式创建能够拟合复杂曲线的模型（参见课程 1 第 2 周的“特征工程与多项式回归”实验）。此外，你还证明了提高多项式的*次数*可以*造成*过拟合（参见课程 1 第 3 周的“过拟合”实验）。现在，让我们运用这些知识，测试自己区分过拟合与欠拟合的能力。

让我们重复训练模型，并在每次迭代中提高多项式次数。为快速简便起见，这里将使用 [scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression) 线性回归模型。

In [ ]:
max_degree = 9
err_train = np.zeros(max_degree)    
err_cv = np.zeros(max_degree)      
x = np.linspace(0,int(X.max()),100)  
y_pred = np.zeros((100,max_degree))  #columns are lines to plot

for degree in range(max_degree):
    lmodel = lin_model(degree+1)
    lmodel.fit(X_train, y_train)
    yhat = lmodel.predict(X_train)
    err_train[degree] = lmodel.mse(y_train, yhat)
    yhat = lmodel.predict(X_cv)
    err_cv[degree] = lmodel.mse(y_cv, yhat)
    y_pred[:,degree] = lmodel.predict(x)
    
optimal_degree = np.argmin(err_cv)+1

<font size="4">让我们绘制结果：</font>

In [ ]:
plt.close("all")
plt_optimal_degree(X_train, y_train, X_cv, y_cv, x, y_pred, x_ideal, y_ideal, 
                   err_train, err_cv, optimal_degree, max_degree)

上图表明，将数据分为两组——模型在其上训练的数据和模型未在其上训练的数据——可以用来判断模型是欠拟合还是过拟合。在我们的示例中，通过提高所用多项式的次数，创建了从欠拟合到过拟合的多种模型。
- 在左图中，实线表示这些模型的预测结果。1 次多项式模型生成一条与极少数据点相交的直线，而最高次数的模型则与每个数据点都贴合得非常紧密。
- 在右图中：
    - 正如预期，训练数据上的误差（蓝色）会随着模型复杂度的提高而下降
    - 随着模型开始顺应数据，交叉验证数据的误差最初会下降；但当模型开始在训练数据上过拟合（无法*泛化*）时，该误差又会上升。
    
值得注意的是，这些示例中的曲线不像课程讲解中可能绘制的那么平滑。显然，分配给每一组的具体数据点会显著改变结果。重要的是整体趋势。

<a name="3.3"></a>
### 3.3 调整正则化
在先前的实验中，您已经使用*正则化*来减少过拟合。与多项式次数类似，可以使用同样的方法调整正则化参数 lambda（$\lambda$）。

下面从高次多项式开始，通过改变正则化参数来演示这一过程。

In [ ]:
lambda_range = np.array([0.0, 1e-6, 1e-5, 1e-4,1e-3,1e-2, 1e-1,1,10,100])
num_steps = len(lambda_range)
degree = 10
err_train = np.zeros(num_steps)    
err_cv = np.zeros(num_steps)       
x = np.linspace(0,int(X.max()),100) 
y_pred = np.zeros((100,num_steps))  #columns are lines to plot

for i in range(num_steps):
    lambda_= lambda_range[i]
    lmodel = lin_model(degree, regularization=True, lambda_=lambda_)
    lmodel.fit(X_train, y_train)
    yhat = lmodel.predict(X_train)
    err_train[i] = lmodel.mse(y_train, yhat)
    yhat = lmodel.predict(X_cv)
    err_cv[i] = lmodel.mse(y_cv, yhat)
    y_pred[:,i] = lmodel.predict(x)
    
optimal_reg_idx = np.argmin(err_cv) 

In [ ]:
plt.close("all")
plt_tune_regularization(X_train, y_train, X_cv, y_cv, x, y_pred, err_train, err_cv, optimal_reg_idx, lambda_range)

上图表明，随着正则化增强，模型会从高方差（过拟合）模型转变为高偏差（欠拟合）模型。右图中的竖线表示 lambda 的最优值。在本例中，多项式次数设置为 10。

<a name="3.4"></a>
### 3.4 获取更多数据：增加训练集大小 (m)
当模型过拟合（高方差）时，收集更多数据可以改善性能。让我们在这里尝试一下。

In [ ]:
X_train, y_train, X_cv, y_cv, x, y_pred, err_train, err_cv, m_range,degree = tune_m()
plt_tune_m(X_train, y_train, X_cv, y_cv, x, y_pred, err_train, err_cv, m_range, degree)

上面的图表明，当模型具有高方差并且发生过拟合时，添加更多样本可以改善性能。请注意左图中的曲线。$m$ 值最大的最后一条曲线是一条位于数据中央的平滑曲线。在右图中，随着样本数量增加，训练集和交叉验证集的性能逐渐收敛到相近的值。请注意，这些曲线不会像课程讲解中所见的那样平滑，这是正常现象。趋势依然很清晰：更多数据能提高泛化能力。

> 请注意，当模型具有高偏差（欠拟合）时，添加更多样本并不能改善性能。

<a name="4"></a>
## 4 - 评估学习算法（神经网络）
上面，你调整了多项式回归模型的各个方面。这里，你将使用神经网络模型。让我们从创建一个分类数据集开始。

<a name="4.1"></a>
### 4.1 数据集
运行下面的单元格生成数据集，并将其拆分为训练集、交叉验证集（CV）和测试集。在本例中，为了突出说明，我们提高了交叉验证数据点的百分比。  

In [ ]:
# Generate and split data set
X, y, centers, classes, std = gen_blobs()

# split the data. Large CV population for demonstration
X_train, X_, y_train, y_ = train_test_split(X,y,test_size=0.50, random_state=1)
X_cv, X_test, y_cv, y_test = train_test_split(X_,y_,test_size=0.20, random_state=1)
print("X_train.shape:", X_train.shape, "X_cv.shape:", X_cv.shape, "X_test.shape:", X_test.shape)

In [ ]:
plt_train_eq_dist(X_train, y_train,classes, X_cv, y_cv, centers, std)

上图左侧显示了数据。图中有六个用颜色区分的簇，同时显示了训练点（圆点）和交叉验证点（三角形）。其中值得关注的是落在模糊位置的点，因为任一簇都可能将它们视为成员。你认为神经网络模型会如何处理？什么情况可以作为过拟合的例子？欠拟合呢？
右侧是一个“理想”模型的示例，或者说，是一个在已知数据来源的情况下可能构建出的模型。各条线表示“等距”边界，即到各中心点的距离相等。值得注意的是，这个模型仍会对整个数据集中约 8% 的数据进行“错误分类”。

<a name="4.2"></a>
### 4.2 通过计算分类误差评估分类模型
此处用于分类模型的评估函数，就是错误预测所占的比例：
$$ J_{cv} =\frac{1}{m}\sum_{i=0}^{m-1} 
\begin{cases}
    1, & \text{if $\hat{y}^{(i)} \neq y^{(i)}$}\\
    0, & \text{otherwise}
\end{cases}
$$

<a name="ex02"></a>
### 练习 2

请补全下面用于计算分类误差的例程。请注意，在本实验中，目标值是类别的索引，并未进行 [one-hot 编码](https://en.wikipedia.org/wiki/One-hot)。

In [ ]:
# UNQ_C2
# GRADED CELL: eval_cat_err
def eval_cat_err(y, yhat):
    """ 
    Calculate the categorization error
    Args:
      y    : (ndarray  Shape (m,) or (m,1))  target value of each example
      yhat : (ndarray  Shape (m,) or (m,1))  predicted value of each example
    Returns:|
      cerr: (scalar)             
    """
    m = len(y)
    incorrect = 0
    for i in range(m):
    ### START CODE HERE ### 
        
    ### END CODE HERE ### 
    
    return(cerr)

In [ ]:
y_hat = np.array([1, 2, 0])
y_tmp = np.array([1, 2, 3])
print(f"categorization error {np.squeeze(eval_cat_err(y_hat, y_tmp)):0.3f}, expected:0.333" )
y_hat = np.array([[1], [2], [0], [3]])
y_tmp = np.array([[1], [2], [1], [3]])
print(f"categorization error {np.squeeze(eval_cat_err(y_hat, y_tmp)):0.3f}, expected:0.250" )

# BEGIN UNIT TEST  
test_eval_cat_err(eval_cat_err)
# END UNIT TEST
# BEGIN UNIT TEST  
test_eval_cat_err(eval_cat_err)
# END UNIT TEST

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
```python
def eval_cat_err(y, yhat):
    """ 
    Calculate the categorization error
    Args:
      y    : (ndarray  Shape (m,) or (m,1))  target value of each example
      yhat : (ndarray  Shape (m,) or (m,1))  predicted value of each example
    Returns:|
      cerr: (scalar)             
    """
    m = len(y)
    incorrect = 0
    for i in range(m):
        if yhat[i] != y[i]:    # @REPLACE
            incorrect += 1     # @REPLACE
    cerr = incorrect/m         # @REPLACE
    return(cerr)                                    
```

<a name="5"></a>
## 5——模型复杂度
下面，您将构建两个模型：一个复杂模型和一个简单模型。您将评估这些模型，以判断它们是否可能过拟合或欠拟合。

### 5.1 复杂模型

<a name="ex03"></a>
### 练习 3
在下面构建一个三层模型：
* 包含 120 个单元、使用 ReLU 激活的全连接层；
* 包含 40 个单元、使用 ReLU 激活的全连接层；
* 包含 6 个单元、使用线性激活（不是 softmax）的全连接层。  
编译时使用：
* 带 `SparseCategoricalCrossentropy` 的损失，请记得使用 `from_logits=True`；
* 学习率为 0.01 的 Adam 优化器。

In [ ]:
# UNQ_C3
# GRADED CELL: model
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)

tf.random.set_seed(1234)
model = Sequential(
    [
        ### START CODE HERE ### 
  
        ### END CODE HERE ### 

    ], name="Complex"
)
model.compile(
    ### START CODE HERE ### 
    loss=None,
    optimizer=None,
    ### END CODE HERE ### 
)

In [ ]:
# BEGIN UNIT TEST
model.fit(
    X_train, y_train,
    epochs=1000
)
# END UNIT TEST

In [ ]:
# BEGIN UNIT TEST
model.summary()

model_test(model, classes, X_train.shape[1]) 
# END UNIT TEST

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
摘要应与以下内容一致（层实例名称可能递增）
```
Model: "Complex"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
=================================================================
L1 (Dense)                   (None, 120)               360       
_________________________________________________________________
L2 (Dense)                   (None, 40)                4840      
_________________________________________________________________
L3 (Dense)                   (None, 6)                 246       
=================================================================
Total params: 5,446
Trainable params: 5,446
Non-trainable params: 0
_________________________________________________________________
```
  <details>
  <summary><font size="3" color="darkgreen"><b>点击查看更多提示</b></font></summary>
  
```python
tf.random.set_seed(1234)
model = Sequential(
    [
        Dense(120, activation = 'relu', name = "L1"),      
        Dense(40, activation = 'relu', name = "L2"),         
        Dense(classes, activation = 'linear', name = "L3")  
    ], name="Complex"
)
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),          
    optimizer=tf.keras.optimizers.Adam(0.01),   
)

model.fit(
    X_train,y_train,
    epochs=1000
)                                  
```

In [ ]:
#make a model for plotting routines to call
model_predict = lambda Xl: np.argmax(tf.nn.softmax(model.predict(Xl)).numpy(),axis=1)
plt_nn(model_predict,X_train,y_train, classes, X_cv, y_cv, suptitle="Complex Model")

该模型非常努力地捕捉每个类别的离群点，结果错误分类了一些交叉验证数据。让我们计算分类误差。

In [ ]:
training_cerr_complex = eval_cat_err(y_train, model_predict(X_train))
cv_cerr_complex = eval_cat_err(y_cv, model_predict(X_cv))
print(f"categorization error, training, complex model: {training_cerr_complex:0.3f}")
print(f"categorization error, cv,       complex model: {cv_cerr_complex:0.3f}")

<a name="5.1"></a>
### 5.1 简单模型
现在，让我们尝试一个简单模型

<a name="ex04"></a>
### 练习 4

下面，构建一个两层模型：
* 包含 6 个单元、使用 ReLU 激活函数的稠密层
* 包含 6 个单元、使用线性激活函数的稠密层。
编译时使用：
* 带 `SparseCategoricalCrossentropy` 的损失函数，记得使用 `from_logits=True`
* 学习率为 0.01 的 Adam 优化器。

In [ ]:
# UNQ_C4
# GRADED CELL: model_s

tf.random.set_seed(1234)
model_s = Sequential(
    [
        ### START CODE HERE ### 
      
        ### END CODE HERE ### 
    ], name = "Simple"
)
model_s.compile(
    ### START CODE HERE ### 
    loss=None,
    optimizer=None,
    ### START CODE HERE ### 
)


In [ ]:
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)

# BEGIN UNIT TEST
model_s.fit(
    X_train,y_train,
    epochs=1000
)
# END UNIT TEST

In [ ]:
# BEGIN UNIT TEST
model_s.summary()

model_s_test(model_s, classes, X_train.shape[1])
# END UNIT TEST

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
摘要应与以下内容一致（层实例名称可能递增）
```
Model: "Simple"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
=================================================================
L1 (Dense)                   (None, 6)                 18        
_________________________________________________________________
L2 (Dense)                   (None, 6)                 42        
=================================================================
Total params: 60
Trainable params: 60
Non-trainable params: 0
_________________________________________________________________
```
  <details>
  <summary><font size="3" color="darkgreen"><b>点击查看更多提示</b></font></summary>
  
```python
tf.random.set_seed(1234)
model_s = Sequential(
    [
        Dense(6, activation = 'relu', name="L1"),            # @REPLACE
        Dense(classes, activation = 'linear', name="L2")     # @REPLACE
    ], name = "Simple"
)
model_s.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),     # @REPLACE
    optimizer=tf.keras.optimizers.Adam(0.01),     # @REPLACE
)

model_s.fit(
    X_train,y_train,
    epochs=1000
)                                   
```

In [ ]:
#make a model for plotting routines to call
model_predict_s = lambda Xl: np.argmax(tf.nn.softmax(model_s.predict(Xl)).numpy(),axis=1)
plt_nn(model_predict_s,X_train,y_train, classes, X_cv, y_cv, suptitle="Simple Model")

这个简单模型表现得相当不错。让我们计算分类误差。

In [ ]:
training_cerr_simple = eval_cat_err(y_train, model_predict_s(X_train))
cv_cerr_simple = eval_cat_err(y_cv, model_predict_s(X_cv))
print(f"categorization error, training, simple model, {training_cerr_simple:0.3f}, complex model: {training_cerr_complex:0.3f}" )
print(f"categorization error, cv,       simple model, {cv_cerr_simple:0.3f}, complex model: {cv_cerr_complex:0.3f}" )

与较复杂的模型相比，我们的简单模型在训练数据上的分类误差略高，但在交叉验证数据上的表现更好。

<a name="6"></a>
## 6 - 正则化
与多项式回归的情况一样，可以应用正则化来减弱更复杂模型的影响。下面来尝试一下。

<a name="ex05"></a>
### 练习 5

重新构建你的复杂模型，但这次要加入正则化。
按照下面的要求构建一个三层模型：
* 包含 120 个单元、使用 ReLU 激活函数和 `kernel_regularizer=tf.keras.regularizers.l2(0.1)` 的密集层
* 包含 40 个单元、使用 ReLU 激活函数和 `kernel_regularizer=tf.keras.regularizers.l2(0.1)` 的密集层
* 包含 6 个单元并使用线性激活函数的密集层。
使用以下配置进行编译：
* 使用 `SparseCategoricalCrossentropy` 的损失函数，记得使用 `from_logits=True`
* 学习率为 0.01 的 Adam 优化器。

In [ ]:
# UNQ_C5
# GRADED CELL: model_r

tf.random.set_seed(1234)
model_r = Sequential(
    [
        ### START CODE HERE ### 
        
        ### START CODE HERE ### 
    ], name= None
)
model_r.compile(
    ### START CODE HERE ### 
    loss=None,
    optimizer=None,
    ### START CODE HERE ### 
)


In [ ]:
# BEGIN UNIT TEST
model_r.fit(
    X_train, y_train,
    epochs=1000
)
# END UNIT TEST

In [ ]:
# BEGIN UNIT TEST
model_r.summary()

model_r_test(model_r, classes, X_train.shape[1]) 
# END UNIT TEST

<details>
  <summary><font size="3" color="darkgreen"><b>单击查看提示</b></font></summary>
    
摘要应与此匹配（层实例名称可能会递增）
```
Model: "ComplexRegularized"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
=================================================================
L1 (Dense)                   (None, 120)               360       
_________________________________________________________________
L2 (Dense)                   (None, 40)                4840      
_________________________________________________________________
L3 (Dense)                   (None, 6)                 246       
=================================================================
Total params: 5,446
Trainable params: 5,446
Non-trainable params: 0
_________________________________________________________________
```
  <details>
  <summary><font size="3" color="darkgreen"><b>单击查看更多提示</b></font></summary>
  
```python
tf.random.set_seed(1234)
model_r = Sequential(
    [
        Dense(120, activation = 'relu', kernel_regularizer=tf.keras.regularizers.l2(0.1), name="L1"), 
        Dense(40, activation = 'relu', kernel_regularizer=tf.keras.regularizers.l2(0.1), name="L2"),  
        Dense(classes, activation = 'linear', name="L3")  
    ], name="ComplexRegularized"
)
model_r.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), 
    optimizer=tf.keras.optimizers.Adam(0.01),                             
)

model_r.fit(
    X_train,y_train,
    epochs=1000
)                                   
``` 

In [ ]:
#make a model for plotting routines to call
model_predict_r = lambda Xl: np.argmax(tf.nn.softmax(model_r.predict(Xl)).numpy(),axis=1)
 
plt_nn(model_predict_r, X_train,y_train, classes, X_cv, y_cv, suptitle="Regularized")

结果看起来与“理想”模型非常相似。让我们检查分类误差。

In [ ]:
training_cerr_reg = eval_cat_err(y_train, model_predict_r(X_train))
cv_cerr_reg = eval_cat_err(y_cv, model_predict_r(X_cv))
test_cerr_reg = eval_cat_err(y_test, model_predict_r(X_test))
print(f"categorization error, training, regularized: {training_cerr_reg:0.3f}, simple model, {training_cerr_simple:0.3f}, complex model: {training_cerr_complex:0.3f}" )
print(f"categorization error, cv,       regularized: {cv_cerr_reg:0.3f}, simple model, {cv_cerr_simple:0.3f}, complex model: {cv_cerr_complex:0.3f}" )

简单模型在训练集上的表现略好于正则化模型，但在交叉验证集上的表现更差。

<a name="7"></a>
## 7——迭代寻找最优正则化值
与在线性回归中一样，您可以尝试多个正则化值。此代码需要几分钟才能运行。如果时间充裕，可以运行并检查结果；如果没有时间，您已经完成了本作业的计分部分！

In [ ]:
tf.random.set_seed(1234)
lambdas = [0.0, 0.001, 0.01, 0.05, 0.1, 0.2, 0.3]
models=[None] * len(lambdas)
for i in range(len(lambdas)):
    lambda_ = lambdas[i]
    models[i] =  Sequential(
        [
            Dense(120, activation = 'relu', kernel_regularizer=tf.keras.regularizers.l2(lambda_)),
            Dense(40, activation = 'relu', kernel_regularizer=tf.keras.regularizers.l2(lambda_)),
            Dense(classes, activation = 'linear')
        ]
    )
    models[i].compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=tf.keras.optimizers.Adam(0.01),
    )

    models[i].fit(
        X_train,y_train,
        epochs=1000
    )
    print(f"Finished lambda = {lambda_}")


In [ ]:
plot_iterate(lambdas, models, X_train, y_train, X_cv, y_cv)

随着正则化程度提高，模型在训练集和交叉验证数据集上的性能会逐渐收敛。对于该数据集和模型，lambda > 0.01 似乎是一个合理的选择。

<a name="7.1"></a>
### 7.1 测试
让我们在测试集上尝试优化后的模型，并与“理想”表现进行比较。

In [ ]:
plt_compare(X_test,y_test, classes, model_predict_s, model_predict_r, centers)

我们的测试集较小，并且似乎包含不少离群点，因此分类误差较高。不过，优化后模型的表现与理想表现相当。

## 恭喜！
您已经熟悉了评估机器学习模型时需要使用的重要工具。具体来说：
* 将数据拆分为训练过和未训练过的集合，可以帮助区分欠拟合与过拟合；
* 创建训练集、交叉验证集和测试集三个数据集，可以：
    * 使用训练集训练参数 $W,B$；
    * 使用交叉验证集调整复杂度、正则化程度和样本数量等模型参数；
    * 使用测试集评估模型在“真实世界”中的表现；
* 比较训练集与交叉验证集上的表现，可以看出模型倾向于过拟合（高方差）还是欠拟合（高偏差）。